# Grammar Scoring Engine for Spoken Responses

This notebook predicts the continuous grammar-MOS label (0–5) from each WAV file. It combines an English ASR transcript with acoustic fluency features. The validation score reported below is out-of-fold (OOF), so it is a more honest benchmark than fitting and scoring on the same 769 rows.

**Metrics:** RMSE is minimized and Pearson correlation is maximized, matching the leaderboard. The final cell writes `submission.csv`.

## Setup

Run this once in Kaggle with Internet enabled, or attach a Kaggle Dataset containing the Faster-Whisper model and point `MODEL_SIZE_OR_PATH` at it. The install line is deliberately separate so an environment that already has the packages does not need network access.

In [ ]:
# Uncomment only if these packages are not already available.
# !pip install -q faster-whisper librosa soundfile scikit-learn pandas seaborn

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

SEED = 42
N_SPLITS = 5
MODEL_SIZE_OR_PATH = 'small.en'  # use a local attached Faster-Whisper model path when Internet is off
# Locate the data both locally and in a Kaggle competition notebook.
local_root = Path('Dataset_Final')
kaggle_roots = [p.parent for p in Path('/kaggle/input').glob('**/train.csv')] if Path('/kaggle/input').exists() else []
DATA_ROOT = local_root if (local_root / 'train.csv').exists() else (kaggle_roots[0] if kaggle_roots else None)
assert DATA_ROOT is not None, 'Could not find train.csv. Attach the competition data under Add Input.'
print(f'Using data at: {DATA_ROOT}')
CACHE_DIR = Path('cache'); CACHE_DIR.mkdir(exist_ok=True)


In [ ]:
train = pd.read_csv(DATA_ROOT / 'train.csv')
test = pd.read_csv(DATA_ROOT / 'test.csv')
sample = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
assert {'filename', 'label'} <= set(train.columns)
assert {'filename', 'label'} <= set(test.columns)
assert train['filename'].is_unique and test['filename'].is_unique
print(f'train: {train.shape}; test: {test.shape}')
if len(sample) != len(test) or set(sample.filename) != set(test.filename):
    print('Warning: sample_submission.csv does not match test.csv. Submission will correctly use test.csv order.')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(train.label, bins=np.arange(-.25, 5.76, .5), ax=ax[0], color='#376996')
ax[0].set(title='Training label distribution', xlabel='Grammar MOS')
sns.boxplot(data=train, y='label', ax=ax[1], color='#91c7b1')
ax[1].set(title='Target range check', ylabel='Grammar MOS')
plt.tight_layout()
train.head()


## Audio preprocessing and features

All audio is converted to mono 16 kHz. We retain duration, energy, silence ratio, zero-crossing rate, spectral centroid, and MFCC summary statistics. These do not claim to measure grammar directly; they capture delivery properties (pauses, consistency, and recording quality) that complement transcript-based evidence.

In [ ]:
def audio_features(path):
    y, sr = librosa.load(path, sr=16_000, mono=True)
    if len(y) == 0:
        return {'duration_s': 0.0}
    rms = librosa.feature.rms(y=y, frame_length=1024, hop_length=512)[0]
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=1024, hop_length=512)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=1024, hop_length=512)[0]
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, n_fft=1024, hop_length=512)
    out = {
        'duration_s': len(y) / sr, 'rms_mean': rms.mean(), 'rms_std': rms.std(),
        'silence_ratio': (rms < max(0.005, np.quantile(rms, .15))).mean(),
        'zcr_mean': zcr.mean(), 'zcr_std': zcr.std(),
        'centroid_mean': centroid.mean(), 'centroid_std': centroid.std(),
    }
    out.update({f'mfcc_{i}_mean': mfcc[i].mean() for i in range(13)})
    out.update({f'mfcc_{i}_std': mfcc[i].std() for i in range(13)})
    return out

def make_audio_frame(frame, split):
    rows = []
    for filename in tqdm(frame.filename, desc=f'{split} acoustic features'):
        rows.append(audio_features(DATA_ROOT / split / filename))
    return pd.DataFrame(rows)

train_audio = make_audio_frame(train, 'train')
test_audio = make_audio_frame(test, 'test')
numeric_cols = list(train_audio.columns)
fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(x=train_audio.duration_s, y=train.label, alpha=.7, ax=ax)
ax.set(title='Duration versus grammar label', xlabel='Duration (seconds)', ylabel='Grammar MOS')
plt.show()


## Speech-to-text (cached)

Grammar is primarily a linguistic property, so transcripts are the central feature. Faster-Whisper is run once and saved in `cache/transcripts.csv`; reruns reuse that file. `beam_size=5` improves transcription consistency without using labels.

In [ ]:
from faster_whisper import WhisperModel
import torch

cache_file = CACHE_DIR / 'transcripts.csv'
all_files = pd.concat([
    train.assign(split='train')[['filename', 'split']],
    test.assign(split='test')[['filename', 'split']]
], ignore_index=True)

if cache_file.exists():
    transcripts = pd.read_csv(cache_file)
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    compute_type = 'float16' if device == 'cuda' else 'int8'
    asr = WhisperModel(MODEL_SIZE_OR_PATH, device=device, compute_type=compute_type)
    records = []
    for row in tqdm(all_files.itertuples(index=False), total=len(all_files), desc='ASR'):
        segments, info = asr.transcribe(str(DATA_ROOT / row.split / row.filename), language='en', beam_size=5, vad_filter=True)
        text = ' '.join(s.text.strip() for s in segments).strip()
        records.append({'filename': row.filename, 'split': row.split, 'transcript': text,
                        'asr_language_probability': info.language_probability})
    transcripts = pd.DataFrame(records)
    transcripts.to_csv(cache_file, index=False)

assert len(transcripts) == len(all_files) and transcripts.transcript.notna().all()
train_text = transcripts.query("split == 'train'").set_index('filename').loc[train.filename].reset_index()
test_text = transcripts.query("split == 'test'").set_index('filename').loc[test.filename].reset_index()
display(train_text[['filename', 'transcript']].head(3))


## Modeling and cross-validation

A word TF–IDF representation captures vocabulary and common grammatical patterns; character n-grams are robust to ASR misspellings and partial words. Acoustic features are standardized and appended. Ridge regression is intentionally regularized because 769 labeled recordings is a small sample. The folds are shuffled and fixed with a seed for reproducibility.

In [ ]:
def linguistic_features(text_train, text_valid):
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=.98, sublinear_tf=True, max_features=30_000)
    char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True, max_features=40_000)
    return hstack([word.fit_transform(text_train), char.fit_transform(text_train)]), hstack([word.transform(text_valid), char.transform(text_valid)]), word, char

y = train.label.to_numpy(float)
texts = train_text.transcript.fillna('').to_numpy()
X_num = train_audio[numeric_cols].to_numpy()
cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof = np.zeros(len(train))
for fold, (tr, va) in enumerate(cv.split(train), 1):
    Xtr_text, Xva_text, _, _ = linguistic_features(texts[tr], texts[va])
    imp = SimpleImputer(strategy='median').fit(X_num[tr])
    scale = StandardScaler().fit(imp.transform(X_num[tr]))
    Xtr = hstack([Xtr_text, csr_matrix(scale.transform(imp.transform(X_num[tr])))])
    Xva = hstack([Xva_text, csr_matrix(scale.transform(imp.transform(X_num[va])))])
    model = Ridge(alpha=18.0).fit(Xtr, y[tr])
    oof[va] = model.predict(Xva)
    print(f'Fold {fold}: RMSE = {mean_squared_error(y[va], oof[va]) ** .5:.4f}')

oof_rmse = mean_squared_error(y, oof) ** .5
oof_pearson = pearsonr(y, oof).statistic
print(f'\nRequired training-data benchmark (5-fold OOF RMSE): {oof_rmse:.4f}')
print(f'5-fold OOF Pearson correlation: {oof_pearson:.4f}')

fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(x=y, y=oof, alpha=.75, ax=ax)
ax.plot([0, 5], [0, 5], '--', color='black')
ax.set(xlim=(0,5), ylim=(0,5), xlabel='True MOS', ylabel='OOF prediction', title=f'OOF predictions | RMSE={oof_rmse:.3f}, r={oof_pearson:.3f}')
plt.show()


## Train on all labeled audio and create submission

After evaluation, vectorizers and model are refit using every training item. Predictions are clipped to the rubric interval `[0, 5]`. The submission uses `test.csv` filenames and row order, which is the authoritative evaluation list.

In [ ]:
test_texts = test_text.transcript.fillna('').to_numpy()
Xtr_text, Xte_text, _, _ = linguistic_features(texts, test_texts)
imp = SimpleImputer(strategy='median').fit(X_num)
scale = StandardScaler().fit(imp.transform(X_num))
Xtr = hstack([Xtr_text, csr_matrix(scale.transform(imp.transform(X_num)))])
Xte_num = test_audio[numeric_cols].to_numpy()
Xte = hstack([Xte_text, csr_matrix(scale.transform(imp.transform(Xte_num)))])
final_model = Ridge(alpha=18.0).fit(Xtr, y)
test_pred = np.clip(final_model.predict(Xte), 0, 5)
submission = pd.DataFrame({'filename': test.filename, 'label': test_pred})
assert len(submission) == len(test)
assert submission.filename.equals(test.filename)
assert submission.label.between(0, 5).all()
submission.to_csv('submission.csv', index=False)
display(submission.head())
print('Wrote submission.csv:', submission.shape)


## Brief report

**Pipeline.** WAV recordings are resampled to 16 kHz mono. Acoustic summaries describe duration, energy, quiet frames, spectral shape, and MFCCs. Faster-Whisper produces an English transcript, which is cached to make experimentation repeatable and affordable. Word and character TF–IDF features plus standardized acoustics feed a regularized Ridge regressor.

**Validation.** Five shuffled K-fold predictions are generated without training on their held-out row. The notebook prints the required training benchmark as 5-fold OOF RMSE and Pearson correlation; these values should be copied into the final competition notes after execution.

**Interpretation and limitations.** Text n-grams provide inspectable linguistic evidence, while acoustic features are supplementary rather than a proxy for grammar. Scores may be affected by ASR errors, accents, or recording conditions. A strong next experiment is to replace `small.en` with an attached `distil-large-v3` Faster-Whisper model, retain exactly the same OOF folds, and compare both metrics before selecting a final submission.